# SANAD v3: النموذج الجديد (التقسيم النظيف) + القائمة المنتقاة
شغّلي الخلايا بالترتيب. الخلية 2 تحتاج نموذج `sanad_retrain_v3/arabert_v3` على Drive (ينتج من نوتبوك التقسيم النظيف). التوكن من Colab Secrets باسم TELEGRAM_TOKEN.


In [ ]:
# الخلية 1: تثبيت المكتبات (شغّلها مرة واحدة)
!pip install -q transformers torch python-telegram-bot nest-asyncio scikit-learn pandas numpy
print("✅ تم التثبيت")

✅ تم التثبيت


In [ ]:
# الخلية 2: تحميل نموذج AraBERT الجديد (المدرَّب على التقسيم النظيف) وتعريف دالة التنبؤ (تعريف واحد فقط)
import os, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
MODEL_PATH = '/content/drive/My Drive/sanad_retrain_v3/arabert_v3'
if not os.path.exists(os.path.join(MODEL_PATH, 'config.json')):
    raise FileNotFoundError('ما لقيت النموذج الجديد. شغّلي نوتبوك التقسيم النظيف أولاً (الخلية 4) أو صحّحي MODEL_PATH')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device).eval()
_LABELS = {0: 'fear', 1: 'joy', 2: 'anger', 3: 'sadness'}   # نفس ترتيب التدريب

def predict_emotion(text):
    """ترجع (العاطفة، الثقة كرقم بين 0 و1، تفاصيل الدرجات). هذا هو التعريف الوحيد في الدفتر."""
    e = tokenizer(str(text), truncation=True, max_length=128, return_tensors='pt').to(device)
    with torch.no_grad():
        p = torch.softmax(model(**e).logits, -1).cpu().numpy()[0]
    scores = {_LABELS[i]: float(p[i]) for i in range(4)}
    best = max(scores, key=scores.get)
    return best, round(scores[best], 4), scores

print('🚀 النموذج جاهز:', predict_emotion('أنا خايف من بكرة')[:2])

Mounted at /content/drive


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

🚀 النموذج جاهز: ('fear', 0.9986)


In [ ]:
# الخلية 3: دوال مساعدة (تنظيف، تحايا، تنقية التشكيل)
import re, random

def clean_arabic_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[\u064B-\u065F\u0670\u06D6-\u06ED]", "", text)
    text = re.sub(r"[إأآٱا]", "ا", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ى", "ي", text)
    return re.sub(r"\s+", " ", text).strip()

GREETINGS_PATTERNS = [r"^اهلا", r"^مرحبتين", r"^مرحبا", r"^السلام عليكم", r"^كيفك",
                      r"^كيف حاسك", r"^صباح الخير", r"^مساء الخير", r"^يا زول"]
SUDANESE_GREETING_RESPONSES = [
    "أهلاً بيك يا زول يا طيب! حبابك حبابك، حاسس بشنو الليلة؟ فضفض لي أنا سامعك.. 🤍",
    "مرحبتين حبابك يا خوي! كيف أمورك؟ أحكي لي الحاصل عليك شنو، بالي كله معاك.",
    "وعليكم السلام والرحمة يا طيب.. أهلاً بيك، حبابك في أي وقت. طمني عليك كيف صحتك ونفسيتك؟",
]

def check_greetings(user_text):
    cleaned = clean_arabic_text(user_text)
    for p in GREETINGS_PATTERNS:
        if re.search(p, cleaned) and len(cleaned.split()) <= 4:
            return True, random.choice(SUDANESE_GREETING_RESPONSES)
    return False, ""

_TAN = {"\u065E": "\u064C", "\u0656": "\u064D", "\u0657": "\u064B"}
def simplify_verse(t):
    """يحوّل التشكيل العثماني إلى تشكيل عادي ويحذف علامات الوقف، بدون حذف الحركات الأساسية."""
    t = t.replace("\u06E1", "\u0652")
    for a, b in _TAN.items():
        t = t.replace(a, b)
    t = t.replace("ٱ", "ا")
    t = re.sub("[\u06D6-\u06DC\u06DF-\u06E0\u06E2-\u06E8\u06EA-\u06ED\u0653]", "", t)   # نُبقي الهمزة (٠٦٥٤/٠٦٥٥) لأن حذفها يكسر كلمات مثل تيأسوا وشيئا
    return re.sub(r"\s+", " ", t).strip()

In [ ]:
# الخلية 4: طبقة الطوارئ (تتحقق من الرسالة قبل أي شيء آخر، حتى قبل التحايا)
# ⚠ أرقام الطوارئ: راجعيها وعدّليها حسب منطقتك. المصدر الذي وجدته يذكر 999 للشرطة والإسعاف والإطفاء في السودان،
#    لكن الخدمة قد تتعطل في مناطق النزاع، لذلك الرسائل تذكر أيضاً "أقرب مستشفى أو شخص قريب".
EMERGENCY_NUMBER = "999"

# الأنماط مكتوبة على النص بعد التطبيع (بدون تشكيل، ة=ه، ى=ي، همزات=ا)
EMERGENCY_RULES = [
    ("self_harm", [
        r"انتحر", r"اقتل نفسي", r"اقتل روحي", r"اذي نفسي", r"اؤذي نفسي", r"ايذاء نفسي",
        r"انهي حياتي", r"اخلص من حياتي", r"عايز اموت", r"عايزه اموت", r"عاوز اموت",
        r"عاوزه اموت", r"ابغي اموت", r"اريد ان اموت", r"ما عايز اعيش", r"ما عايزه اعيش",
        r"مابقي عايز اعيش", r"اتمني الموت", r"اتمني اموت", r"اشنق نفسي", r"اقطع شرايني",
    ]),
    ("life_weary", [
        r"(تعب|زهق|زهج|مل|كره|يئس|بقيت زهق)\w* (من )?(الحياه|حياتي|العيش|عيشتي)", r"ما عندي رغبه في (الحياه|العيش)",
        r"ما في (فايده|معني) (من|في) (حياتي|الحياه)", r"حياتي ما لها (معني|قيمه|فايده)", r"لو ما كنت موجود", r"لو ما اتولدت",
        r"احسن لو (اختفيت|ما كنت)", r"بقيت عبء علي (الناس|اهلي|الكل)", r"ما بستاهل اعيش", r"ما بستاهل الحياه",
        r"(تعبان|تعبانه|تعبت|زهقت|زهقان|زهقانه|مليت|ملان|ملانه) من الدنيا", r"ما عندي سبب (اعيش|استمر)", r"(ياريت|يا ريت|ياريتني) ما كنت موجود",
    ]),
    ("medical", [
        r"الم (شديد|في صدري|في قلبي|في معدتي|في بطني|بطني|معدتي)", r"الم صدر", r"وجع صدر",
        r"ما قادر اتنفس", r"ما قادره اتنفس", r"لا اقدر اتنفس", r"ضيق تنفس", r"نزيف", r"ينزف", r"تنزف",
        r"اغماء", r"اغمي عليه", r"اغمي عليها", r"فقد الوعي", r"فاقد الوعي", r"جلطه", r"تسمم",
        r"ازمه (صحيه|قلبيه)", r"نوبه (قلبيه|صرع)", r"حاله ولاده",
    ]),
    ("violence", [
        r"حرامي", r"حراميه", r"حرامين", r"سرقه", r"سارق", r"سرقوا", r"يسرقون", r"لصوص", r"(?<![ء-ي])لص(?![ء-ي])",
        r"مسلح", r"سلاح", r"اعتداء", r"اعتدي", r"تهديد بالقتل", r"يهددني", r"يهددوني", r"خطف", r"اختطاف",
        r"بيضربني", r"يضربني", r"ضربني", r"يضربون", r"ضربوني", r"بيضربوني", r"(?<![ء-ي])عنف(?![ء-ي])", r"اغتصاب",
    ]),
    ("fire", [
        r"حريق", r"حرايق", r"دخان", r"انفجار", r"تسرب غاز", r"ولعت", r"اشتعل", r"بتحترق", r"يحترق",
    ]),
    ("accident", [
        r"(?<![ء-ي])(ال|وال|بال)?حادث[هة]?(?![ء-ي])", r"اصطدم", r"انقلبت", r"دهس",
    ]),
]

EMERGENCY_MESSAGES = {
    "self_harm": (
        "🤍 كلامك مهم عندي ومهتم بسلامتك.\n"
        "لو في خطر إنك تؤذي نفسك الآن:\n"
        "• اتصل{|ي} بالطوارئ (__EMG__) أو اذهب{|ي} لأقرب مستشفى فوراً.\n"
        "• {ما تبقى|ما تبقي} براك، نادي أقرب زول ليك (أهل، جيران، صديق) واقعد{|ي} معاه.\n"
        "• ابعد{|ي} عن أي شيء ممكن تؤذي بيه نفسك.\n"
        "أنا بوت وما أقدر أحل مكان مختص، لكن أنا هنا أسمعك، وتقدر{|ي} تحكي لي الحاصل."
    ),
    "life_weary": (
        "🤍 حاسّ بتعب كبير في كلامك، وشكراً إنك حكيت لي. تعب الحياة ده تقيل وما لازم تشيل الحمل براك.\n"
        "• لو بتفكر{|ي} تؤذي نفسك أو تنهي حياتك، اتصل{|ي} بالطوارئ (__EMG__) أو اذهب{|ي} لأقرب مستشفى فوراً، ونادي أقرب زول ليك واقعد{|ي} معاه.\n"
        "• لو ما كده، كلّم{|ي} زول بتثق{|ي} فيه أو مختص نفسي، وده ما عيب.\n"
        "أنا بوت وما أقدر أحل مكان مختص، لكن أنا هنا معاك وأسمعك."
    ),
    "medical": (
        "🚑 الحالة دي ممكن تكون طبية عاجلة.\n"
        "• اتصل{|ي} بالإسعاف/الطوارئ (__EMG__) أو اذهب{|ي} لأقرب مستشفى فوراً.\n"
        "• نادي أي زول قريب منك يقعد معاك، وما تكون{|ي} براك.\n"
        "• لو في صعوبة تنفس أو نزيف شديد أو فقدان وعي، ما تنتظر{|ي}.\n"
        "سلامتك أولاً، ربنا يلطف بيك ويكتب ليك العافية 🤍"
    ),
    "violence": (
        "🚔 سلامتك الآن هي الأهم.\n"
        "• لو في خطر مباشر، اتصل{|ي} بالشرطة (__EMG__) فوراً.\n"
        "• اقفل{|ي} الأبواب، وابعد{|ي} عن مكان الخطر، وما تواجه{|ي} أي زول بنفسك.\n"
        "• نادي الجيران أو أقرب زول تثق فيه.\n"
        "ربنا يحفظك ويحفظ أهلك 🤍"
    ),
    "fire": (
        "🔥 الأولوية: اطلع{|ي} من المكان فوراً.\n"
        "• اتصل{|ي} بالإطفاء/الدفاع المدني (__EMG__).\n"
        "• ما ترجع{|ي} تجيب{|ي} حاجات، وانتبه{|ي} من الدخان (ابقَ{|ي} منخفضاً).\n"
        "• نبّه{|ي} الناس حواليك.\n"
        "ربنا يسلمك 🤍"
    ),
    "accident": (
        "🚑 لو في إصابات أو حادث:\n"
        "• اتصل{|ي} بالإسعاف/الشرطة (__EMG__).\n"
        "• ما تحرّك{|ي} المصاب إلا لو في خطر مباشر.\n"
        "• خلّ{|ي} زول قريب معاك لحدّ ما توصل المساعدة.\n"
        "سلامتك أولاً 🤍"
    ),
}

EMERGENCY_MESSAGES = {k: v.replace("__EMG__", EMERGENCY_NUMBER) for k, v in EMERGENCY_MESSAGES.items()}

class SafetyLayer:
    def __init__(self):
        self.rules = [(cat, [re.compile(p) for p in pats]) for cat, pats in EMERGENCY_RULES]

    def check(self, text):
        norm = clean_arabic_text(text)
        for cat, pats in self.rules:          # الترتيب مقصود: إيذاء النفس أولاً ثم الطبي ثم العنف ثم الحريق
            if any(p.search(norm) for p in pats):
                return True, cat, EMERGENCY_MESSAGES[cat]
        return False, "", ""

safety_layer = SafetyLayer()

In [ ]:
# الخلية 5: بنوك الردود (لهجة سودانية) وقائمة الآيات المنتقاة (مراجع فقط، النص يُجلب ويُتحقق منه في الخلية 6)
# صيغة {مذكر|مؤنث} تتحوّل حسب جنس المستخدم. عدّلي أي جملة تحبي تغيّريها هنا.
import json
BANKS = json.loads(r'''{
 "fear": {
  "open": [
   "حاسّ بالخوف دا يا زول، وما ببراك في الموضوع.. أنا معاك هنا.",
   "الخوف شعور تقيل وبتعب القلب، لكن الحمد لله إنك قلت لي، وده خطوة كويسة.",
   "اطمّن{|ي} يا {خوي|أختي}، الخوف دا طبيعي وبيعدّي بإذن الله.",
   "فاهم قلقك وشايل معاك همّه شوية.. خلينا نهدّي الأنفاس ونتذكر حاجة تطمّن.",
   "ما تخاف{|ي} براك، ربنا قريب منك أكتر من ما تتخيل{|ي}.",
   "قلبك لمن بيخاف بيحتاج كلمة تثبّته.. خليني أشاركك واحدة منها.",
   "أنا سامعك وحاسّ بيك.. الخوف ده ما بيدوم، وربنا رحيم بيك.",
   "الله يطمّن قلبك يا {طيب|طيبة}.. جيب{|ي} نفس عميق وتعال{|ي} نقرأ حاجة تريّح.",
   "كتر الخوف أحياناً بيخلينا نحس إنو الدنيا ضاقت، لكن ربنا واسع وفرجه قريب.",
   "يا زول ما تشيل هم أكتر من طاقتك.. في ربّ بحفظك وبسمعك.",
   "أنا هنا معاك، وما حا أخلّيك براك.. وخلينا نتذكر كلام ربنا اللي بيطمّن.",
   "بالي معاك والله، وخوفك ده مفهوم.. اسمع{|ي} الآية دي:"
  ],
  "bridge": [
   "وربنا بيقول في كتابه:",
   "اقرأ{|ي} معاي الكلام ده بهدوء:",
   "في القرآن كلام بيطمّن القلب، اسمع{|ي}:",
   "خلّ{|ي} الآية دي تدخل قلبك:",
   "ودي آية تتذكرها كل ما الخوف يجيك:",
   "سمع{|ي} ربنا شنو بقول:",
   "وربنا بخاطبنا في حالات زي دي:",
   "اتأمّل{|ي} الآية دي شوية:",
   "اقرأ{|ي} الآية دي بقلب هادي:",
   "ومن الكلام اللي يثبّت القلب:"
  ],
  "close": [
   "ربنا يحفظك ويثبّت قلبك، وأنا معاك لو عايز{|ة} تحكي أكتر.",
   "جيب{|ي} نفس عميق.. الأمر بإذن الله ما حا يكون زي ما بتخاف.",
   "فضفض{|ي} لي أكتر، أنا سامعك وما مستعجل.",
   "ربنا يطمّن قلبك ويكفيك شر ما تخاف منه.",
   "لو الخوف زاد، كلّم{|ي} زول بتثق فيه، ما تشيل{|ي} الحمل براك.",
   "الله معاك، وأنا برضو هنا.. قول لي شنو اللي مخوّفك أكتر؟",
   "إن شاء الله الأمور بتتيسّر، وربنا ما بخذل عبده.",
   "خلّ{|ي} قلبك مع ربنا، وهو أكرم من إنو يتركك.",
   "اشرب{|ي} موية وخُد{|ي} نفس، وبإذن الله الصباح يجي بخير.",
   "ربنا يرزقك الأمان والسكينة يا {خوي|أختي}.",
   "وحاول{|ي} ما تقعد{|ي} براك الليلة، وأنا برضو موجود لو احتجت{|ي}.",
   "إن شاء الله يعدّي الخوف ده كأنو ما كان."
  ],
  "dua": [
   "اللهم اجعل لنا من كل همٍّ فرجاً ومن كل ضيقٍ مخرجاً.",
   "اللهم اجعل في قلوبنا سكينة تغنينا عن كل خوف.",
   "اللهم احفظنا بحفظك الذي لا يضيع ما استودعته.",
   "اللهم طمّن قلوبنا وثبّت أقدامنا.",
   "اللهم اكفنا بما شئت وكيف شئت إنك على كل شيء قدير.",
   "اللهم لا سهل إلا ما جعلته سهلاً وأنت تجعل الحزن إذا شئت سهلاً."
  ]
 },
 "sadness": {
  "open": [
   "يا زول يا {حبيبي|حبيبتي}.. فاهم حزنك وشايل معاك جزء منه، وما ببراك.",
   "الحزن بيقعد في الصدر تقيل، والحمد لله إنك فضفضت لي.",
   "قلبي معاك والله.. الحزن دا حقيقي وله حقّه، وما في زول بلومك عليه.",
   "أنا سامعك، وخلّي دموعك تنزل لو عايز{|ة}.. دي رحمة من ربنا.",
   "الدنيا أحياناً بتتقل علينا، لكن ربنا أرحم بينا من أنفسنا.",
   "مرّات الحزن بيجي من غير سبب واضح، وده برضو بيعدّي بإذن الله.",
   "يا {خوي|أختي} الليلة أنا هنا ليك.. وما براك.",
   "وجعك دا مفهوم، وما بتحتاج{|ي} تتظاهر{|ي} بإنك بخير قدامي.",
   "ربنا ما بحمّل نفس فوق طاقتها.. وأنا معاك خطوة خطوة.",
   "الحزن ما بيبقى للأبد.. لكن الليلة خلّينا نمسّك الأمل سوا.",
   "ربنا يجبر بخاطرك ويطيّب قلبك.. اسمع{|ي} الكلام ده:",
   "تعبان{|ة} من الحمل دا؟ ضع{|ي} شوية منو عندي، وخلّ{|ي} الباقي لله."
  ],
  "bridge": [
   "وربنا بقول في كتابه:",
   "اسمع{|ي} الآية دي بقلبك:",
   "وفي القرآن مواساة للقلوب المكسورة:",
   "خلّ{|ي} الكلام ده يطبطب عليك:",
   "ومن الآيات اللي بتتذكرها في الضيق:",
   "اقرأ{|ي} معاي بهدوء:",
   "وربنا بخاطب القلوب الحزينة:",
   "وفي كلامه سبحانه:",
   "خُد{|ي} نفس وتأمّل{|ي}:",
   "ومن رحمة ربنا إنو قال:"
  ],
  "close": [
   "ربنا يطيّب بخاطرك ويعوّضك خير، وأنا معاك لو عايز{|ة} تحكي أكتر.",
   "خُد{|ي} وقتك، ما في استعجال في الشفاء.. وأنا هنا.",
   "لو الحزن طوّل معاك، كلّم{|ي} زول قريب أو مختص، ما عيب.",
   "اشرب{|ي} موية وارتاح{|ي} شوية.. وبعدين لو عايز{|ة} نكمّل كلامنا.",
   "إن شاء الله بعد العسر يسر، وفرج ربنا قريب.",
   "قول لي: شنو الحاجة الأكتر موجعاك الليلة؟ أنا سامعك.",
   "ربنا يرحمك ويرحم ضعفك ويجبر كسرك.",
   "يا رب يهوّن عليك ويربط على قلبك.",
   "خلّ{|ي} رجاءك في ربنا كبير، فهو أكرم الأكرمين.",
   "الحمد لله إنك ما براك، وأنا هنا بالسمع.",
   "كل نفس بتعدّي في حياتها بأيام تقيلة، ودي بتعدّي.. صدّق{|ي}.",
   "ربنا يملا قلبك سكينة وطمأنينة."
  ],
  "dua": [
   "اللهم اجبر كسرنا وفرّج كربنا وارحم ضعفنا.",
   "اللهم لا تجعل الدنيا أكبر همّنا.",
   "اللهم اجعل القرآن ربيع قلوبنا ونور صدورنا وجلاء أحزاننا وذهاب همومنا.",
   "اللهم أبدلنا بعد الحزن فرحاً وبعد الضيق سعة.",
   "اللهم ارزقنا الصبر والرضا والسكينة.",
   "اللهم اكتب لنا العافية والفرج القريب."
  ]
 },
 "anger": {
  "open": [
   "فاهم إنو الغضب ده اجاك من حاجة وجعتك، وحقّك تحس بيهو.",
   "الغضب شعور بيحصل لكل زول.. المهم كيف نتعامل معاهو.",
   "اهدأ{|ي} يا {خوي|أختي}، خُد{|ي} نفس عميق وعدّ{|ي} لعشرة.",
   "أنا سامعك.. قول{|ي} لي الحصل، وما بحكم عليك.",
   "قبل ما تردّ{|ي} على أي زول، اقعد{|ي} شوية واشرب{|ي} موية.",
   "الغضب لو اتركناه بياكل في القلب، فخلّينا نهدّيهو بذكر ربنا.",
   "حقّك تزعل{|ي} وحقّك تتألم{|ي}، لكن ما نخلّي الغضب يقرّر عنّك.",
   "يا زول اللحظة الأولى هي الأصعب.. لو عدّت بتهدا بإذن الله.",
   "أحياناً الغضب بيكون وجع متخفّي.. شنو اللي ضايقك فعلاً؟",
   "ربنا يهدّي قلبك.. وخلّيني أشاركك كلام بيطفّي النار دي:",
   "فاهمك والله، لكن خلّينا نتذكر قدوتنا في الحلم:",
   "الغضب نار صغيرة، والحلم موية بطفيها.. اسمع{|ي} معاي:"
  ],
  "bridge": [
   "وربنا بقول لينا:",
   "اقرأ{|ي} معاي بهدوء:",
   "وفي القرآن وصف للناس الطيبين:",
   "ومن الكلام اللي بيهدّي القلب:",
   "اسمع{|ي} الآية دي:",
   "وربنا بيوصّينا:",
   "تأمّل{|ي} الآية دي:",
   "ومن آداب ديننا في الغضب:",
   "اقرأ{|ي} الآية دي ببطء:",
   "وخلّ{|ي} الكلام ده يبرّد قلبك:"
  ],
  "close": [
   "ربنا يهدّي قلبك ويرزقك الحلم، وأنا هنا لو عايز{|ة} تفضفض{|ي}.",
   "خُد{|ي} نفس.. وإذا قدرت{|ي}، ابعد{|ي} عن الموقف لحظات لحدّ ما تهدا.",
   "ولو قلت{|ي} كلمة وندمت{|ي} عليها، الاعتذار شجاعة.",
   "ما تتصرّف{|ي} وانت{|ي} غاضب{|ة}.. اشرب{|ي} موية وتوضّأ{|ي} لو قدرت{|ي}.",
   "الله يعينك، وبإذن الله الأمور بتتصلّح.",
   "وصلّ{|ي} على النبي ﷺ يهدا قلبك.",
   "لو حابّ{|ة}، احكي لي شنو الحصل بالتفصيل وأنا سامعك.",
   "قوّتك الحقيقية في إنك تمسك نفسك، وإنت{|ي} قدّها.",
   "ربنا يعوّضك عن اللي أزعلك بخير منو.",
   "خلّ{|ي} قلبك أبيض، فده أريح ليك قبل أي زول.",
   "جرّب{|ي} تمشي{|ي} شوية أو تغيّر{|ي} المكان، بيساعد.",
   "الغضب بيعدّي، لكن الكلمة الطيبة بتبقى."
  ],
  "dua": [
   "اللهم إني أسألك كلمة الحق في الرضا والغضب.",
   "اللهم ارزقنا الحلم وحسن الخلق.",
   "اللهم ألّف بين قلوبنا وأصلح ذات بيننا.",
   "اللهم طهّر قلوبنا من الغلّ والحقد.",
   "اللهم اهدِنا لأحسن الأخلاق لا يهدي لأحسنها إلا أنت.",
   "اللهم اجعلنا ممن يكظمون الغيظ ويعفون عن الناس."
  ]
 },
 "joy": {
  "open": [
   "ما شاء الله تبارك الله! فرحتك دي بتفرّحني أنا كمان.",
   "الحمد لله على النعمة دي! ربنا يتمّها ويديمها عليك.",
   "ياااا سلام! خبر بيفرّح القلب والله.",
   "مبروك ليك يا {طيب|طيبة}! ربنا يبارك فيك وفي كل خطواتك.",
   "ربنا يديم عليك الفرحة والسرور.. حاسّ بسعادتك.",
   "بالله ده كلام بيشرّح الصدر! الله يزيدك من فضله.",
   "فرحك دا نعمة، وخلّينا نشارك الله الحمد عليها.",
   "الله يبارك فيك ويحرسك من العين.. فرحت لي كتير.",
   "الأيام الحلوة دي بتبقى ذكرى جميلة، الله يدوّمها.",
   "من قلبي: الله يتمّها بخير ويرزقك أكتر.",
   "نعمة تستاهل شكر.. وخلّيني أشاركك آية جميلة فيها:",
   "الحمد لله دائماً وأبداً.. خلّينا نحتفل بالطريقة الحلوة دي:"
  ],
  "bridge": [
   "وربنا بقول عن الشكر:",
   "اسمع{|ي} الآية دي اللي بتناسب فرحتك:",
   "وفي القرآن ذكر للنعمة والشكر:",
   "تأمّل{|ي} الآية دي:",
   "وخلّ{|ي} الآية دي تزيد فرحك بركة:",
   "ومن جميل كلام ربنا:",
   "اقرأ{|ي} معاي:",
   "وفي كتاب الله:",
   "ودي آية بتتذكرها وإنت{|ي} فرحان{|ة}:",
   "ربنا بقول:"
  ],
  "close": [
   "ربنا يزيدك من فضله ويديم عليك النعمة، وشارك{|ي}ني تفاصيل الفرحة لو حابّ{|ة}!",
   "لا تنسى{|ي} تشكر{|ي} ربنا وتشارك{|ي} الفرحة مع ناسك.",
   "الله يبارك لك فيها ويجعلها خير.",
   "حافظ{|ي} على النعمة بالشكر، والشكر بيزيدها.",
   "فرحتك دي تستاهل احتفال بسيط مع الأحباب.",
   "قول لي أكتر عن اللي فرّحك!",
   "الحمد لله، ربنا يكتب ليك التوفيق دائماً.",
   "الله يتمّ عليك بخير ويديم عليك الستر.",
   "تذكّر{|ي} الأيام دي لمن تحس{|ي} بتعب، وتشكر{|ي} ربنا.",
   "ربنا يسعدك في الدنيا والآخرة يا {خوي|أختي}.",
   "الله يجعل أيامك كلها فرح وبركة.",
   "أنا فرحان{|ة} ليك من قلبي."
  ],
  "dua": [
   "اللهم لك الحمد كما ينبغي لجلال وجهك وعظيم سلطانك.",
   "اللهم أوزعنا أن نشكر نعمتك التي أنعمت علينا.",
   "اللهم ما أصبح بنا من نعمة فمنك وحدك لا شريك لك فلك الحمد ولك الشكر.",
   "اللهم بارك لنا فيما أعطيتنا وزدنا من فضلك.",
   "اللهم اجعلنا من الشاكرين.",
   "اللهم أتمّ علينا نعمتك وبارك لنا فيها."
  ]
 },
 "harm": {
  "open": [
   "كلامك وجعني.. إنك تتأذى من زول ده ما بسيط، وحقّك تحسّ بالغضب والوجع.",
   "فاهمك، لمن زول يأذينا بيجي غضب وجرح، وما محتاج{|ة} تبرّر إحساسك لي.",
   "أنا سامعك وبصدّقك.. الغضب هنا طبيعي ومفهوم.",
   "ما بحكم على الحصل، المهم إني معاك ومهتم بيك.",
   "الأذى بيوجع، وغضبك رد فعل طبيعي على وجعك.",
   "شكراً إنك حكيت لي.. دي خطوة شجاعة."
  ],
  "bridge": [
   "وفي القرآن آية بتهدّي القلب في اللحظات الصعبة:",
   "خُد{|ي} نفس.. وتأمّل{|ي} الآية دي:",
   "وربنا بقول:",
   "اسمع{|ي} الآية دي بهدوء:"
  ],
  "close": [
   "أهم حاجة سلامتك: هل إنت{|ي} بخير وآمن{|ة} هسع؟",
   "لو الأذى بيتكرر أو بيخوّفك، كلّم{|ي} زول بتثق{|ي} فيه أو جهة مختصة، وما تتحمّل{|ي} براك.",
   "ما لازم تقرّر{|ي} حاجة وإنت{|ي} غاضب{|ة}.. خُد{|ي} وقتك، وأنا معاك.",
   "لو حابّ{|ة}، احكي لي أكتر الحصل، وأنا سامعك بدون حكم.",
   "حقّك تكون{|ي} في أمان وتتحترم{|ي}."
  ],
  "dua": [
   "اللهم احفظنا بحفظك واكفنا شر ما نخاف.",
   "اللهم اشرح صدورنا ويسّر أمورنا.",
   "اللهم اجبر كسرنا وارزقنا السكينة."
  ]
 },
 "neutral": {
  "open": [
   "أنا سامعك وفاهم كلامك يا زول.. تحب نكمّل؟",
   "شكراً إنك حكيت لي.. أنا هنا ما مستعجل.",
   "حاسّ إنو في حاجة في بالك.. خلّينا نمشي بهدوء.",
   "أنا معاك.. وخلّيني أشاركك كلمة بتهدّي القلب.",
   "فضفض{|ي} لي براحتك، وما في حكم.",
   "كلامك وصلني، وخليني أرد عليك بكلمة تفيد القلب.",
   "أحياناً ما بنعرف نسمّي إحساسنا، وده طبيعي.. خلّينا نتأمل شوية.",
   "أنا هنا.. وربنا بقول لينا حاجة جميلة في الحالات دي:"
  ],
  "bridge": [
   "وربنا بقول:",
   "اسمع{|ي} الآية دي:",
   "ومن كلام ربنا:",
   "تأمّل{|ي} معاي:",
   "اقرأ{|ي} بهدوء:",
   "وفي كتاب الله:"
  ],
  "close": [
   "فضفض{|ي} لي أكتر لو حابّ{|ة}، أنا هنا عشان أسمعك.",
   "قول لي أكتر عن إحساسك، عشان أفهمك كويس.",
   "ربنا يطمّن قلبك ويسهّل أمورك.",
   "لو حابّ{|ة} تحكي أكتر، أنا سامعك.",
   "الله يبارك في يومك ويعينك.",
   "وإنت{|ي} حرّ{|ة}، خُد{|ي} وقتك."
  ],
  "dua": [
   "اللهم اجعل القرآن ربيع قلوبنا.",
   "اللهم اشرح صدورنا ويسّر أمورنا.",
   "اللهم اهدنا وسدّدنا.",
   "اللهم ارزقنا السكينة والطمأنينة."
  ]
 }
}''')
FEM_MARKERS = json.loads(r'''["خايفه", "زعلانه", "حزينه", "فرحانه", "غضبانه", "تعبانه", "قلقانه", "مرهقه", "وحيده", "مكتئبه", "مبسوطه", "سعيده", "متوتره", "مرعوبه", "مصدومه", "مجروحه", "محتاجه", "عايزه", "عاوزه", "قادره", "ضايقه", "منكسره", "مخنوقه", "محبطه", "متضايقه", "زهجانه", "جعانه", "نادمه", "مشتاقه", "متحمسه", "فخوره"]''')
CURATED_REFS = json.loads(r'''{
 "fear": [
  {
   "s": 20,
   "a": 46,
   "key": "لا تخافا انني معكما",
   "theme": "معيّة الله وسمعه ورؤيته",
   "trig": [
    "تهديد",
    "يهددني",
    "ظلم",
    "مدير",
    "مسؤول"
   ],
   "story": "الآية دي لما أرسل ربنا موسى وهارون عليهما السلام لفرعون وهم خايفين من بطشه، فطمّنهم إنه معاهم يسمع ويرى.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 9,
   "a": 40,
   "key": "لا تحزن ان الله معنا",
   "theme": "الله معك في الشدة",
   "trig": [
    "وحيد",
    "وحدي",
    "براي",
    "معزول",
    "ما معاي"
   ],
   "story": "قالها النبي ﷺ لصاحبه أبي بكر وهما في الغار في الهجرة والمشركين قريبين منهم.",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "اذ يقول لصاحبه لا تحزن ان الله معنا",
    "اذ يقول لصاحبه لا تحزن ان الله معنا"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 2,
   "a": 286,
   "key": "لا يكلف الله نفسا الا وسعها",
   "theme": "ما فوق الطاقة",
   "trig": [
    "ما قادر",
    "ما قادره",
    "طاقه",
    "ضغط",
    "تعب",
    "مسؤوليه",
    "ما بقدر"
   ],
   "story": "",
   "gen": true,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": [
    "لا يكلف الله نفسا الا وسعها",
    "لا يكلف الله نفسا الا وسعها"
   ],
   "taf_skip": 0,
   "taf_n": 1
  },
  {
   "s": 3,
   "a": 173,
   "key": "حسبنا الله ونعم الوكيل",
   "theme": "التوكل والاكتفاء بالله",
   "trig": [
    "ناس",
    "كلام الناس",
    "عدو",
    "حسد",
    "يهددني",
    "يخوفني"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 13,
   "a": 28,
   "key": "الا بذكر الله تطمئن القلوب",
   "theme": "طمأنينة القلب بالذكر",
   "trig": [
    "قلق",
    "توتر",
    "اضطراب",
    "مضطرب",
    "قلبي",
    "خفقان"
   ],
   "story": "",
   "gen": true,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 65,
   "a": 2,
   "key": "ومن يتق الله يجعل له مخرجا",
   "theme": "المخرج والرزق",
   "trig": [
    "رزق",
    "مال",
    "فقر",
    "وظيفه",
    "شغل",
    "عاطل",
    "بطاله",
    "ديون",
    "دين",
    "ايجار",
    "ضيق",
    "مشكله"
   ],
   "story": "",
   "gen": false,
   "to": 3,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "ومن يتق الله يجعل له مخرجا",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 10,
   "a": 62,
   "key": "لا خوف عليهم ولا هم يحزنون",
   "theme": "لا خوف على أولياء الله",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 41,
   "a": 30,
   "key": "الا تخافوا ولا تحزنوا",
   "theme": "بشارة الملائكة بالأمان",
   "trig": [
    "موت",
    "احتضار",
    "قبر"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 39,
   "a": 36,
   "key": "اليس الله بكاف عبده",
   "theme": "كفاية الله لعبده",
   "trig": [
    "يهددني",
    "ناس",
    "كلام الناس",
    "عدو",
    "حسد"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "اليس الله بكاف عبده",
    "اليس الله بكاف عبده"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 12,
   "a": 64,
   "key": "فالله خير حافظا",
   "theme": "حفظ الله لمن نحب",
   "trig": [
    "ولدي",
    "ابني",
    "بتي",
    "طفل",
    "رضيع",
    "ولد",
    "بنت",
    "عيالي",
    "سفر",
    "مسافر",
    "غايب",
    "اهلي",
    "امي",
    "ابوي"
   ],
   "story": "قالها يعقوب عليه السلام لما طلب منه أبناؤه أن يرسل معهم أخاهم بنيامين وقد فقد يوسف من قبل، فسلّم أمره لله وقال إن الله خير حافظا.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "فالله خير حافظا وهو ارحم الراحمين",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 28,
   "a": 7,
   "key": "ولا تخافي ولا تحزني",
   "theme": "طمأنة الأم على ولدها",
   "trig": [
    "ولدي",
    "ابني",
    "بتي",
    "طفل",
    "رضيع",
    "ولد",
    "بنت",
    "عيالي",
    "بنتي",
    "امي"
   ],
   "story": "دي لما طمّن ربنا أم موسى وهي خايفة على رضيعها وأوحى لها ما تخاف وما تحزن.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 20,
   "a": 68,
   "key": "لا تخف انك انت الاعلي",
   "theme": "الثبات عند المواجهة",
   "trig": [
    "مواجهه",
    "امتحان",
    "مقابله",
    "منافسه",
    "اختبار",
    "نتيجه"
   ],
   "story": "قالها ربنا لموسى عليه السلام لما خاف يوم مواجهة السحرة.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 27,
   "a": 10,
   "key": "لا تخف اني لا يخاف لدي المرسلون",
   "theme": "الأمان عند الله",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "يا موسي لا تخف اني لا يخاف لدي المرسلون",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 94,
   "a": 5,
   "key": "فان مع العسر يسرا",
   "theme": "اليسر بعد العسر",
   "trig": [
    "ضيق",
    "صعب",
    "صعبه"
   ],
   "story": "",
   "gen": true,
   "to": 6,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 33,
   "a": 3,
   "key": "وتوكل علي الله وكفي بالله وكيلا",
   "theme": "التوكل",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 9,
   "a": 51,
   "key": "قل لن يصيبنا الا ما كتب الله لنا",
   "theme": "الرضا بما كتب الله",
   "trig": [
    "مستقبل",
    "بكره",
    "غدا",
    "قدر",
    "نصيب",
    "بكرة"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 2,
   "a": 38,
   "key": "فلا خوف عليهم ولا هم يحزنون",
   "theme": "الأمان لمن اتبع الهدى",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "فاما ياتينكم مني هدي فمن تبع هداي فلا خوف عليهم ولا هم يحزنون",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 11,
   "a": 123,
   "key": "فاعبده وتوكل عليه",
   "theme": "التوكل بعد العمل",
   "trig": [
    "قرار",
    "اختيار"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": [
    "فاعبده وتوكل عليه",
    "وما ربك بغافل عما تعملون"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 14,
   "a": 12,
   "key": "وما لنا الا نتوكل علي الله",
   "theme": "التوكل مع الصبر على الأذى",
   "trig": [
    "اذي",
    "ظلم"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 5,
   "a": 23,
   "key": "وعلي الله فتوكلوا ان كنتم مؤمنين",
   "theme": "التوكل",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "وعلي الله فتوكلوا ان كنتم مؤمنين",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 48,
   "a": 4,
   "key": "هو الذي انزل السكينه في قلوب المؤمنين",
   "theme": "السكينة",
   "trig": [
    "قلق",
    "رعب",
    "ذعر",
    "نوبه",
    "ارتجاف"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": [
    "هو الذي انزل السكينه في قلوب المؤمنين",
    "ايمانهم"
   ],
   "taf_skip": 0,
   "taf_n": 1
  },
  {
   "s": 42,
   "a": 19,
   "key": "الله لطيف بعباده",
   "theme": "لطف الله",
   "trig": [
    "رزق",
    "مال",
    "فقر",
    "وظيفه",
    "شغل",
    "عاطل",
    "بطاله",
    "ديون",
    "دين",
    "ايجار"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "الله لطيف بعباده يرزق من يشاء",
    "يرزق من يشاء"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 51,
   "a": 22,
   "key": "وفي السماء رزقكم وما توعدون",
   "theme": "الرزق مضمون",
   "trig": [
    "رزق",
    "مال",
    "فقر",
    "وظيفه",
    "شغل",
    "عاطل",
    "بطاله",
    "ديون",
    "دين",
    "ايجار"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 11,
   "a": 6,
   "key": "وما من دابه في الارض الا علي الله رزقها",
   "theme": "الرزق على الله",
   "trig": [
    "رزق",
    "مال",
    "فقر",
    "وظيفه",
    "شغل",
    "عاطل",
    "بطاله",
    "ديون",
    "دين",
    "ايجار"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 29,
   "a": 60,
   "key": "الله يرزقها واياكم",
   "theme": "الله يرزق الجميع",
   "trig": [
    "رزق",
    "مال",
    "فقر",
    "وظيفه",
    "شغل",
    "عاطل",
    "بطاله",
    "ديون",
    "دين",
    "ايجار"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 6,
   "a": 17,
   "key": "وان يمسسك الله بضر فلا كاشف له الا هو",
   "theme": "كاشف الضر",
   "trig": [
    "مرض",
    "مريض",
    "وجع",
    "علاج",
    "عمليه",
    "مستشفي",
    "ابتلاء",
    "بلاء"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 26,
   "a": 62,
   "key": "كلا ان معي ربي سيهدين",
   "theme": "الثقة بالهداية والنجاة",
   "trig": [
    "حيران",
    "ضايع",
    "محتار",
    "محتاره",
    "ما عارف",
    "ما عارفه",
    "مش عارف",
    "تايه",
    "مشتت",
    "قرار",
    "طريق"
   ],
   "story": "قالها موسى عليه السلام لما شاف قومه فرعون وراهم والبحر قدامهم، ورغم كده قال ربي معي وسيهديني.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 21,
   "a": 69,
   "key": "يا نار كوني بردا وسلاما",
   "theme": "النجاة بإذن الله",
   "trig": [
    "حريق",
    "نار"
   ],
   "story": "لما رموا إبراهيم عليه السلام في النار، أمر ربنا النار تبقى بردا وسلاما عليه.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 3,
   "a": 175,
   "key": "فلا تخافوهم وخافون ان كنتم مؤمنين",
   "theme": "الخوف من الناس",
   "trig": [
    "ناس",
    "كلام الناس",
    "عدو",
    "حسد",
    "يخوفني"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  }
 ],
 "sadness": [
  {
   "s": 93,
   "a": 3,
   "key": "ما ودعك ربك وما قلي",
   "theme": "الله ما تخلّى عنك",
   "trig": [
    "وحيد",
    "وحيده",
    "متروك",
    "ما حد",
    "تخلوا",
    "تخلي",
    "وحدي"
   ],
   "story": "نزلت تطمّن النبي ﷺ لما انقطع عنه الوحي فترة وحزن.",
   "gen": true,
   "to": 5,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 2,
   "taf_n": null
  },
  {
   "s": 94,
   "a": 1,
   "key": "الم نشرح لك صدرك",
   "theme": "شرح الصدر",
   "trig": [
    "ضيق",
    "صدري",
    "مخنوق",
    "خنقه",
    "مكتئب",
    "اكتئاب"
   ],
   "story": "",
   "gen": false,
   "to": 4,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 12,
   "a": 86,
   "key": "انما اشكو بثي وحزني الي الله",
   "theme": "الشكوى إلى الله",
   "trig": [
    "شكوي",
    "احكي",
    "فضفض",
    "قلبي",
    "موجوع"
   ],
   "story": "قالها يعقوب عليه السلام لما فقد يوسف وأخاه، ولجأ لربه بحزنه.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 12,
   "a": 87,
   "key": "ولا تياسوا من روح الله",
   "theme": "لا تيأس من رحمة الله",
   "trig": [
    "يأس",
    "ياس",
    "ما في امل",
    "مافي امل",
    "استسلم",
    "تعبت",
    "خلاص"
   ],
   "story": "قالها يعقوب عليه السلام لأبنائه وهو ما فقد الأمل برغم طول الغياب.",
   "gen": false,
   "to": 87,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "ولا تياسوا من روح الله",
    "ولا تياسوا من روح الله"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 39,
   "a": 53,
   "key": "لا تقنطوا من رحمه الله",
   "theme": "رحمة الله وسعت الذنوب",
   "trig": [
    "ذنب",
    "ذنوب",
    "غلطه",
    "ندمان",
    "ندم",
    "اخطات",
    "غلطت"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 2,
   "a": 153,
   "key": "استعينوا بالصبر والصلاه ان الله مع الصابرين",
   "theme": "الصبر والصلاة",
   "trig": [
    "صبر",
    "تعبان",
    "تعبانه",
    "مرهق"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 2,
   "a": 155,
   "key": "انا لله وانا اليه راجعون",
   "theme": "عند المصيبة",
   "trig": [
    "مات",
    "توفي",
    "وفاه",
    "موت",
    "فقدت",
    "رحل",
    "عزاء",
    "الله يرحمه",
    "الله يرحمها",
    "حادث",
    "خسارة"
   ],
   "story": "",
   "gen": false,
   "to": 157,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 21,
   "a": 83,
   "key": "اني مسني الضر وانت ارحم الراحمين",
   "theme": "دعاء المريض",
   "trig": [
    "مرض",
    "مريض",
    "وجع",
    "علاج",
    "عمليه",
    "مستشفي",
    "ابتلاء",
    "بلاء"
   ],
   "story": "دي دعوة أيوب عليه السلام لما أصابه المرض والبلاء مدة طويلة، وصبر ولجأ لربه.",
   "gen": false,
   "to": 84,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 21,
   "a": 87,
   "key": "لا اله الا انت سبحانك اني كنت من الظالمين",
   "theme": "دعاء الكرب",
   "trig": [
    "هم",
    "غم",
    "كرب",
    "ضيق",
    "ظلمه",
    "محبوس",
    "مكتئب"
   ],
   "story": "دعاء يونس عليه السلام وهو في بطن الحوت في الظلمات، فنجّاه الله.",
   "gen": false,
   "to": 88,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "فنادي في الظلمات ان لا اله الا انت",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 3,
   "a": 139,
   "key": "ولا تهنوا ولا تحزنوا وانتم الاعلون",
   "theme": "لا تضعف ولا تحزن",
   "trig": [
    "ضعيف",
    "محبط",
    "احباط",
    "انكسر"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 16,
   "a": 127,
   "key": "واصبر وما صبرك الا بالله",
   "theme": "الصبر بالله",
   "trig": [
    "صبر",
    "اصبر",
    "ما قادر اصبر"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 2,
   "a": 216,
   "key": "وعسي ان تكرهوا شيئا وهو خير لكم",
   "theme": "الخير الخفي",
   "trig": [
    "رسبت",
    "فشل",
    "فاتني",
    "ما حصل",
    "رفضوا",
    "رفض",
    "خسرت",
    "فصل",
    "طردوني",
    "ما اتقبلت"
   ],
   "story": "",
   "gen": true,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "وعسي ان تكرهوا شيئا وهو خير لكم",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 57,
   "a": 22,
   "key": "ما اصاب من مصيبه في الارض",
   "theme": "كل شيء بقدر",
   "trig": [
    "فاتني",
    "فاتتني",
    "خسرت",
    "ضاع",
    "ضاعت"
   ],
   "story": "",
   "gen": false,
   "to": 23,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": [
    "ما اصاب من مصيبه في الارض",
    "ولا تفرحوا بما اتاكم"
   ],
   "taf_skip": 0,
   "taf_n": 3
  },
  {
   "s": 12,
   "a": 18,
   "key": "فصبر جميل والله المستعان",
   "theme": "الصبر الجميل",
   "trig": [
    "صبر",
    "خذلوني",
    "غدروا",
    "كذبوا علي"
   ],
   "story": "قالها يعقوب عليه السلام لما أخبره أبناؤه بخبر يوسف وهو متألم.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "فصبر جميل",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 12,
   "a": 83,
   "key": "عسي الله ان ياتيني بهم جميعا",
   "theme": "الأمل بالله رغم الغياب",
   "trig": [
    "غايب",
    "مسافر",
    "اشتاق",
    "اشتياق",
    "بعيد",
    "فراق",
    "هجره",
    "غربه"
   ],
   "story": "قالها يعقوب عليه السلام وهو بعيد عن أبنائه ومازال يرجو الله.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "فصبر جميل عسي الله ان ياتيني بهم جميعا",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 28,
   "a": 13,
   "key": "كي تقر عينها ولا تحزن",
   "theme": "جبر الخاطر",
   "trig": [
    "ولدي",
    "ابني",
    "بتي",
    "طفل",
    "رضيع",
    "ولد",
    "بنت",
    "عيالي",
    "فراق",
    "اشتياق"
   ],
   "story": "في قصة موسى عليه السلام لما رده الله لأمه عشان تفرح وما تحزن.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": [
    "فرددناه الي امه كي تقر عينها ولا تحزن ولتعلم ان وعد الله حق",
    "وعد الله حق"
   ],
   "taf_skip": 0,
   "taf_n": 1
  },
  {
   "s": 65,
   "a": 7,
   "key": "سيجعل الله بعد عسر يسرا",
   "theme": "اليسر بعد العسر",
   "trig": [
    "ضيق",
    "ديون",
    "فقر",
    "مال"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "لا يكلف الله نفسا الا ما اتاها سيجعل الله بعد عسر يسرا",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 2,
   "a": 186,
   "key": "فاني قريب اجيب دعوه الداع اذا دعان",
   "theme": "الله قريب يجيب الدعاء",
   "trig": [
    "وحيد",
    "ادعو",
    "دعاء",
    "دعيت",
    "ما استجاب",
    "ما بيستجيب"
   ],
   "story": "",
   "gen": true,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 40,
   "a": 60,
   "key": "ادعوني استجب لكم",
   "theme": "الدعاء",
   "trig": [
    "ادعو",
    "دعاء",
    "دعيت"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "وقال ربكم ادعوني استجب لكم",
    "ادعوني استجب لكم"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 27,
   "a": 62,
   "key": "امن يجيب المضطر اذا دعاه ويكشف السوء",
   "theme": "إجابة المضطر",
   "trig": [
    "ضيق",
    "كرب",
    "ضايق",
    "ضايقه",
    "مضطر"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "امن يجيب المضطر اذا دعاه ويكشف السوء",
    "ويكشف السوء"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 3,
   "a": 200,
   "key": "اصبروا وصابروا ورابطوا",
   "theme": "الصبر والمصابرة",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 39,
   "a": 10,
   "key": "انما يوفي الصابرون اجرهم بغير حساب",
   "theme": "أجر الصابرين",
   "trig": [
    "صبر",
    "صبرت",
    "تعبت"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "انما يوفي الصابرون اجرهم بغير حساب",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 64,
   "a": 11,
   "key": "ما اصاب من مصيبه الا باذن الله ومن يؤمن بالله يهد قلبه",
   "theme": "هداية القلب عند المصيبة",
   "trig": [
    "مات",
    "توفي",
    "وفاه",
    "موت",
    "فقدت",
    "رحل",
    "عزاء",
    "الله يرحمه",
    "الله يرحمها",
    "مصيبه",
    "ابتلاء"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 15,
   "a": 97,
   "key": "ولقد نعلم انك يضيق صدرك بما يقولون",
   "theme": "ضيق الصدر من كلام الناس",
   "trig": [
    "كلام الناس",
    "انتقاد",
    "ضيق",
    "استهزاء",
    "سخريه",
    "تنمر"
   ],
   "story": "",
   "gen": false,
   "to": 98,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 10,
   "a": 57,
   "key": "وشفاء لما في الصدور",
   "theme": "شفاء الصدور",
   "trig": [
    "صدري",
    "ضيق",
    "اكتئاب",
    "حزن",
    "مخنوق"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  }
 ],
 "anger": [
  {
   "s": 3,
   "a": 134,
   "key": "والكاظمين الغيظ والعافين عن الناس",
   "theme": "كظم الغيظ والعفو",
   "trig": [
    "غضب",
    "غضبان",
    "زعلان",
    "عصبي"
   ],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 42,
   "a": 37,
   "key": "واذا ما غضبوا هم يغفرون",
   "theme": "المغفرة عند الغضب",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": false,
   "notaf": false,
   "ex": [
    "واذا ما غضبوا هم يغفرون",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 42,
   "a": 43,
   "key": "ولمن صبر وغفر ان ذلك لمن عزم الامور",
   "theme": "الصبر والمغفرة",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 7,
   "a": 199,
   "key": "خذ العفو وامر بالعرف",
   "theme": "خذ العفو",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 7,
   "a": 200,
   "key": "واما ينزغنك من الشيطان نزغ فاستعذ بالله",
   "theme": "الاستعاذة عند الغضب",
   "trig": [
    "غضب",
    "عصبي",
    "اتعصب",
    "اعصابي",
    "افقد اعصابي",
    "صوتي",
    "صرخت"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 41,
   "a": 34,
   "key": "ادفع بالتي هي احسن فاذا الذي بينك وبينه عداوه",
   "theme": "ادفع بالتي هي أحسن",
   "trig": [
    "عدو",
    "عداوه",
    "مشكله مع",
    "خصام",
    "زعل",
    "خصمت",
    "اذاني",
    "كلمني بسوء"
   ],
   "story": "",
   "gen": false,
   "to": 35,
   "forgive": true,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 1,
   "taf_n": null
  },
  {
   "s": 25,
   "a": 63,
   "key": "واذا خاطبهم الجاهلون قالوا سلاما",
   "theme": "الرد بالسلام على الجاهل",
   "trig": [
    "شتمني",
    "سبني",
    "اهانني",
    "كلام جارح",
    "اهانه",
    "سب"
   ],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 16,
   "a": 126,
   "key": "ولئن صبرتم لهو خير للصابرين",
   "theme": "الصبر خير",
   "trig": [
    "انتقم",
    "اخذ حقي",
    "اضربه",
    "ارد عليه"
   ],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": false,
   "notaf": true,
   "ex": [
    "ولئن صبرتم لهو خير للصابرين",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 15,
   "a": 85,
   "key": "فاصفح الصفح الجميل",
   "theme": "الصفح الجميل",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": false,
   "notaf": true,
   "ex": [
    "فاصفح الصفح الجميل",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 3,
   "a": 159,
   "key": "فبما رحمه من الله لنت لهم",
   "theme": "اللين والرحمة",
   "trig": [
    "شغل",
    "زملاء",
    "فريق",
    "مدير",
    "طلاب",
    "عيالي"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 23,
   "a": 96,
   "key": "ادفع بالتي هي احسن السيئه",
   "theme": "ادفع السيئة بالحسنة",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 17,
   "a": 53,
   "key": "وقل لعبادي يقولوا التي هي احسن",
   "theme": "القول الحسن",
   "trig": [
    "كلام",
    "صوتي",
    "جادلت",
    "نقاش",
    "صرخت",
    "شتمت"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 2,
   "a": 263,
   "key": "قول معروف ومغفره خير من صدقه",
   "theme": "كلمة طيبة ومغفرة",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 49,
   "a": 10,
   "key": "انما المؤمنون اخوه فاصلحوا بين اخويكم",
   "theme": "الإصلاح بين الإخوة",
   "trig": [
    "اخوي",
    "اخي",
    "اختي",
    "صاحبي",
    "صديقي",
    "خلاف",
    "خصام",
    "زعلان من"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 20,
   "a": 44,
   "key": "فقولا له قولا لينا",
   "theme": "القول اللين حتى مع الظالم",
   "trig": [
    "ظالم",
    "متكبر",
    "مدير",
    "مسؤول",
    "متسلط"
   ],
   "story": "قالها ربنا لموسى وهارون وهما ذاهبين لفرعون الطاغية: كلماه بلين.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 16,
   "a": 125,
   "key": "وجادلهم بالتي هي احسن",
   "theme": "الجدال بالحسنى",
   "trig": [
    "جدال",
    "نقاش",
    "خلاف",
    "جادلت"
   ],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": true,
   "notaf": true,
   "ex": [
    "وجادلهم بالتي هي احسن",
    "وجادلهم بالتي هي احسن"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 28,
   "a": 55,
   "key": "ولكم اعمالكم سلام عليكم",
   "theme": "الإعراض عن اللغو",
   "trig": [
    "لغو",
    "كلام فارغ",
    "استفزني",
    "استفزاز",
    "اهانه"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 1,
   "taf_n": null
  },
  {
   "s": 25,
   "a": 72,
   "key": "واذا مروا باللغو مروا كراما",
   "theme": "المرور الكريم باللغو",
   "trig": [
    "لغو",
    "كلام فارغ",
    "استفزاز"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 45,
   "a": 14,
   "key": "قل للذين امنوا يغفروا للذين لا يرجون ايام الله",
   "theme": "المغفرة",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": false,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 31,
   "a": 17,
   "key": "واصبر علي ما اصابك ان ذلك من عزم الامور",
   "theme": "الصبر على الأذى",
   "trig": [
    "اذي",
    "ظلم",
    "اذاني"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 11,
   "a": 115,
   "key": "واصبر فان الله لا يضيع اجر المحسنين",
   "theme": "الصبر وأجر المحسنين",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 60,
   "a": 7,
   "key": "عسي الله ان يجعل بينكم وبين الذين عاديتم منهم موده",
   "theme": "عسى أن يجعل الله مودة",
   "trig": [
    "عدو",
    "عداوه",
    "خصام",
    "قطيعه",
    "مقاطعه"
   ],
   "story": "",
   "gen": false,
   "forgive": true,
   "narrow": true,
   "notaf": true,
   "ex": [
    "عسي الله ان يجعل بينكم وبين الذين عاديتم منهم مودة",
    "مودة"
   ],
   "taf_skip": 0,
   "taf_n": null
  }
 ],
 "joy": [
  {
   "s": 14,
   "a": 7,
   "key": "لئن شكرتم لازيدنكم",
   "theme": "الشكر يزيد النعمة",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "واذ تاذن ربكم لئن شكرتم لازيدنكم",
    "لازيدنكم"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 10,
   "a": 58,
   "key": "قل بفضل الله وبرحمته فبذلك فليفرحوا",
   "theme": "الفرح بفضل الله",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 93,
   "a": 11,
   "key": "واما بنعمه ربك فحدث",
   "theme": "التحدث بالنعمة",
   "trig": [
    "نجحت",
    "نجاح",
    "تخرجت",
    "تخرج",
    "وظيفه",
    "ترقيه",
    "خطوبه",
    "زواج",
    "عرس",
    "مولود"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 27,
   "a": 19,
   "key": "رب اوزعني ان اشكر نعمتك",
   "theme": "دعاء الشكر",
   "trig": [
    "نجحت",
    "نجاح",
    "تخرجت",
    "مولود",
    "اتفاجات",
    "نعمه"
   ],
   "story": "قالها سليمان عليه السلام لما سمع كلام النملة وتبسم ضاحكا وشكر ربه.",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 27,
   "a": 40,
   "key": "هذا من فضل ربي ليبلوني",
   "theme": "من فضل ربي",
   "trig": [
    "نجحت",
    "نجاح",
    "وظيفه",
    "ترقيه",
    "فزت",
    "فوز"
   ],
   "story": "قالها سليمان عليه السلام لما شاف عرش ملكة سبأ حاضرا بين يديه بفضل الله.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": [
    "فلما راه مستقرا عنده قال هذا من فضل ربي ليبلوني",
    "ام اكفر"
   ],
   "taf_skip": 2,
   "taf_n": 2
  },
  {
   "s": 16,
   "a": 18,
   "key": "وان تعدوا نعمه الله لا تحصوها",
   "theme": "نعم الله لا تحصى",
   "trig": [
    "الحمد لله",
    "نعمه",
    "نعم"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": [
    "وان تعدوا نعمه الله لا تحصوها",
    "لا تحصوها"
   ],
   "taf_skip": 0,
   "taf_n": 1
  },
  {
   "s": 55,
   "a": 13,
   "key": "فباي الاء ربكما تكذبان",
   "theme": "آلاء الله",
   "trig": [
    "جميل",
    "روعه",
    "جمال"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": 1
  },
  {
   "s": 34,
   "a": 13,
   "key": "اعملوا ال داود شكرا",
   "theme": "العمل شكرا",
   "trig": [
    "شغل",
    "وظيفه",
    "ترقيه",
    "نجاح"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "اعملوا ال داود شكرا وقليل من عبادي الشكور",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 31,
   "a": 12,
   "key": "ان اشكر لله ومن يشكر فانما يشكر لنفسه",
   "theme": "الشكر لله",
   "trig": [],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": [
    "ان اشكر لله ومن يشكر فانما يشكر لنفسه",
    "لنفسه"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 2,
   "a": 152,
   "key": "فاذكروني اذكركم واشكروا لي",
   "theme": "اذكروني أذكركم",
   "trig": [],
   "story": "",
   "gen": true,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 16,
   "a": 53,
   "key": "وما بكم من نعمه فمن الله",
   "theme": "كل نعمة من الله",
   "trig": [
    "نعمه",
    "نعم",
    "الحمد لله"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": [
    "وما بكم من نعمه فمن الله",
    "فمن الله"
   ],
   "taf_skip": 0,
   "taf_n": 1
  },
  {
   "s": 46,
   "a": 15,
   "key": "رب اوزعني ان اشكر نعمتك",
   "theme": "دعاء الشكر والصلاح",
   "trig": [
    "مولود",
    "ولد",
    "بنت",
    "ابني",
    "بتي",
    "عمري",
    "عيد ميلاد",
    "اربعين"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": [
    "حتي اذا بلغ اشده وبلغ اربعين سنه",
    "واصلح لي في ذريتي"
   ],
   "taf_skip": 2,
   "taf_n": 1
  },
  {
   "s": 10,
   "a": 64,
   "key": "لهم البشري في الحياه الدنيا",
   "theme": "البشرى في الدنيا",
   "trig": [
    "بشاره",
    "خبر حلو",
    "خبر سار"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 12,
   "a": 101,
   "key": "رب قد اتيتني من الملك وعلمتني من تاويل الاحاديث",
   "theme": "دعاء يوسف عند تمام النعمة",
   "trig": [
    "تخرجت",
    "لقاء",
    "اجتمعنا",
    "اجتماع",
    "رجع",
    "رجعوا",
    "عودة"
   ],
   "story": "من دعاء يوسف عليه السلام بعد ما جمعه الله بأبيه وإخوته ومكّن له.",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 2,
   "a": 172,
   "key": "كلوا من طيبات ما رزقناكم واشكروا لله",
   "theme": "شكر الرزق",
   "trig": [
    "اكل",
    "عزومه",
    "وليمه",
    "عرس",
    "مناسبه",
    "رزق"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "كلوا من طيبات ما رزقناكم واشكروا لله",
    "واشكروا لله"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 16,
   "a": 114,
   "key": "فكلوا مما رزقكم الله حلالا طيبا واشكروا نعمت الله",
   "theme": "شكر الرزق الحلال",
   "trig": [
    "رزق",
    "اكل",
    "وليمه",
    "عرس"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": [
    "فكلوا مما رزقكم الله حلالا طيبا واشكروا نعمت الله",
    "نعمت الله"
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 35,
   "a": 34,
   "key": "الحمد لله الذي اذهب عنا الحزن",
   "theme": "الحمد لله الذي أذهب الحزن",
   "trig": [
    "كان حزين",
    "فرج",
    "انفرجت",
    "زال همي",
    "ذهب الحزن"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "الحمد لله الذي اذهب عنا الحزن ان ربنا لغفور شكور",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 1,
   "a": 2,
   "key": "الحمد لله رب العالمين",
   "theme": "الحمد لله رب العالمين",
   "trig": [
    "الحمد لله"
   ],
   "story": "",
   "gen": true,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 18,
   "a": 46,
   "key": "المال والبنون زينه الحياه الدنيا",
   "theme": "الباقيات الصالحات",
   "trig": [
    "مال",
    "ولد",
    "ارباح",
    "كسبت"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 40,
   "a": 65,
   "key": "فادعوه مخلصين له الدين الحمد لله رب العالمين",
   "theme": "الحمد لله",
   "trig": [
    "الحمد لله"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": false,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 76,
   "a": 9,
   "key": "انما نطعمكم لوجه الله لا نريد منكم جزاء ولا شكورا",
   "theme": "الكرم والإخلاص",
   "trig": [
    "صدقه",
    "عزومه",
    "كرم",
    "وليمه",
    "تبرعت"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": true,
   "ex": [
    "انما نطعمكم لوجه الله لا نريد منكم جزاء ولا شكورا",
    ""
   ],
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 27,
   "a": 73,
   "key": "وان ربك لذو فضل علي الناس",
   "theme": "فضل الله على الناس",
   "trig": [
    "فضل",
    "نعمه"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": false,
   "notaf": true,
   "ex": null,
   "taf_skip": 0,
   "taf_n": null
  },
  {
   "s": 31,
   "a": 14,
   "key": "ان اشكر لي ولوالديك",
   "theme": "شكر الله والوالدين",
   "trig": [
    "امي",
    "ابوي",
    "والدي",
    "والدتي",
    "اهلي"
   ],
   "story": "",
   "gen": false,
   "forgive": false,
   "narrow": true,
   "notaf": false,
   "ex": [
    "ان اشكر لي ولوالديك",
    "ولوالديك"
   ],
   "taf_skip": 0,
   "taf_n": 1
  }
 ]
}''')
print("بنوك الردود:", {k: len(v["open"]) * len(v["bridge"]) * len(v["close"]) for k, v in BANKS.items()}, "(تركيبات ممكنة لكل شعور)")
print("آيات مقترحة:", {k: len(v) for k, v in CURATED_REFS.items()})

بنوك الردود: {'fear': 1440, 'sadness': 1440, 'anger': 1440, 'joy': 1440, 'harm': 120, 'neutral': 288} (تركيبات ممكنة لكل شعور)
آيات مقترحة: {'fear': 29, 'sadness': 25, 'anger': 22, 'joy': 23}


In [ ]:
# الخلية 6: بناء القائمة المنتقاة (تحتاج إنترنت، تعمل في Colab). تجلب نص الآية والتفسير الميسر وتتحقق من كل مرجع.
# أي مرجع تفشل عبارته المفتاحية يُرفض ويُطبع سببه، فلا تدخل آية خطأ في البوت.
# ---- بناء القائمة المنتقاة: جلب نص الآية والتفسير من مصدر خارجي والتحقق من صحة كل مرجع بعبارة مفتاحية ----
import re, json, time

def norm_key(t):
    t = clean_arabic_text(t)
    return re.sub(r"[^\w\s]", "", t).replace("ـ", "")

def _skel_words(t):
    # هيكل الكلمة: حذف حروف المد والهمزات حتى لا يفشل التحقق بسبب اختلاف الإملاء العثماني/المبسّط
    return [re.sub("[اوىيءأإآؤئةه]", "", w) for w in norm_key(t).split()]

def key_in_text(key, text):
    k, t = _skel_words(key), _skel_words(text)
    return any(t[i:i + len(k)] == k for i in range(len(t) - len(k) + 1))

def strip_basmala(text):
    toks = text.split()
    if len(toks) > 4 and norm_key(" ".join(toks[:4])) == "بسم الله الرحمن الرحيم":
        return " ".join(toks[4:])
    return text

def short_surah(name):
    n = re.sub(r"[ً-ٰٟۖ-ۭ]", "", str(name)).strip()
    return re.sub(r"^سور[ةه]\s*", "", n).strip()

def fetch_ayah_api(s, a):
    """يجلب الآية (عثماني + مبسّط) وتفسيرها الميسر من alquran.cloud. يحتاج إنترنت (Colab فيه)."""
    import requests
    url = f"https://api.alquran.cloud/v1/ayah/{s}:{a}/editions/quran-uthmani,quran-simple,ar.muyassar"
    last = None
    for attempt in range(4):
        try:
            r = requests.get(url, timeout=25)
            r.raise_for_status()
            by = {d["edition"]["identifier"]: d for d in r.json()["data"]}
            return {"uth": by["quran-uthmani"]["text"], "simple": by["quran-simple"]["text"],
                    "tafsir": by["ar.muyassar"]["text"], "surah": by["quran-uthmani"]["surah"]["name"]}
        except Exception as ex:
            last = ex
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"فشل جلب {s}:{a}: {last}")

def build_curated(refs_by_emotion, fetch, pause=0.15):
    cache, out, rejected = {}, {}, []
    def get(s, a):
        if (s, a) not in cache:
            cache[(s, a)] = fetch(s, a)
            time.sleep(pause)
        return cache[(s, a)]
    for emo, entries in refs_by_emotion.items():
        out[emo] = []
        for i, e in enumerate(entries):
            s, a, to = e["s"], e["a"], e.get("to", e["a"])
            try:
                parts = [get(s, k) for k in range(a, to + 1)]
            except Exception as ex:
                rejected.append({"emotion": emo, "ref": f"{s}:{a}", "reason": f"فشل الجلب: {ex}"})
                continue
            fix = (lambda t, k: strip_basmala(t) if (k == 1 and s not in (1, 9)) else t)
            uth = " ".join(fix(p["uth"], a + j) for j, p in enumerate(parts))
            simple = " ".join(fix(p["simple"], a + j) for j, p in enumerate(parts))
            if not key_in_text(e["key"], simple):
                rejected.append({"emotion": emo, "ref": f"{s}:{a}", "reason": "العبارة المفتاحية غير موجودة في النص (المرجع قد يكون خطأ)"})
                continue
            ref = f"{short_surah(parts[0]['surah'])}: {a}" + (f"-{to}" if to != a else "")
            out[emo].append({"id": f"{emo}-{s}-{a}", "ref": ref, "verse": uth, "tafsir": " ".join(dict.fromkeys(p["tafsir"] for p in parts)),
                             "theme": e["theme"], "trig": e["trig"], "story": e["story"], "gen": e["gen"], "narrow": e.get("narrow", False)})
    return {"by_emotion": out, "rejected": rejected}


import os, json, pandas as pd
CURATED_PATH = "/content/drive/My Drive/sanad_curated.json"   # يُحفظ في Drive ويُعاد استخدامه
if os.path.exists(CURATED_PATH):
    curated = json.load(open(CURATED_PATH, encoding="utf-8"))
    print("تم تحميل القائمة المحفوظة:", CURATED_PATH)
else:
    curated = build_curated(CURATED_REFS, fetch_ayah_api)
    os.makedirs(os.path.dirname(CURATED_PATH), exist_ok=True)
    json.dump(curated, open(CURATED_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    rows = [dict(الشعور=emo, المرجع=e["ref"], الموضوع=e["theme"], الآية=simplify_verse(e["verse"]) if "simplify_verse" in globals() else e["verse"],
                 التفسير=e["tafsir"], القصة=e["story"], وسوم=" ، ".join(e["trig"]), سياق_ضيق=e["narrow"])
            for emo, lst in curated["by_emotion"].items() for e in lst]
    pd.DataFrame(rows).to_csv("/content/sanad_curated_review.csv", index=False, encoding="utf-8-sig")
print("آيات مقبولة:", {k: len(v) for k, v in curated["by_emotion"].items()})
print("مرفوضة:", len(curated["rejected"]))
for r in curated["rejected"]:
    print("  ✗", r["emotion"], r["ref"], "-", r["reason"])

تم تحميل القائمة المحفوظة: /content/drive/My Drive/sanad_curated.json
آيات مقبولة: {'fear': 29, 'sadness': 25, 'anger': 29, 'joy': 24}
مرفوضة: 0


In [ ]:
# الخلية 7: المحرك (اختيار الآية حسب الموضوع، تركيب الرد، وصيغة التذكير/التأنيث)
# ---- المحرك: اختيار الآية حسب الموضوع، وتركيب رد متنوع، ومراعاة تذكير/تأنيث المستخدم ----
import re, random, collections

_GEN_RE = re.compile(r"\{([^{}|]*)\|([^{}|]*)\}")
def fill_gender(text, fem):
    return _GEN_RE.sub(lambda m: m.group(2) if fem else m.group(1), text)

_FEM_SET = set(FEM_MARKERS)
FEM_USERS = set()
def detect_female(text):
    words = set(clean_arabic_text(text).split())
    return bool(words & _FEM_SET)

def dedupe_sentences(t):
    seen, out = set(), []
    for s in re.split(r"(?<=[.؟!])\s+", re.sub(r"\s+", " ", t).strip()):
        k = s.strip()
        if k and k not in seen:
            seen.add(k); out.append(k)
    return " ".join(out)

def trim_tafsir(t, limit=380, hard=620):
    t = dedupe_sentences(t)
    if len(t) <= limit:
        return t
    k = t[:limit].rfind(".")
    if k > 120:
        return t[:k + 1]
    k = t.find(".", limit)          # لا نقطع الجملة في منتصفها: نكمل لنهايتها إن لم تطل
    if 0 < k <= hard:
        return t[:k + 1]
    cut = t[:limit]
    j = max(cut.rfind("،"), cut.rfind("؛"))
    return (cut[:j] if j > 120 else cut.rstrip()) + " …"

def _variants(trig):
    t = clean_arabic_text(trig)
    return [t] + [p + t for p in ("ال", "و", "ب", "ل")]

def _skel(w):
    return re.sub("[اوىيءأإآؤئةه]", "", norm_key(w))

def cut_excerpt(verse, start_phrase, end_phrase):
    """يعرض جزءاً من الآية: يطابق العبارات بهيكل الحروف فلا يتأثر بالإملاء العثماني. إن لم يجد يرجع الآية كاملة (None)."""
    words = verse.split()
    seq = [(i, _skel(w)) for i, w in enumerate(words)]
    seq = [(i, s) for i, s in seq if s]
    sk = [s for _, s in seq]
    def find(phrase, begin=0):
        p = [_skel(w) for w in phrase.split()]
        p = [s for s in p if s]
        for j in range(begin, len(sk) - len(p) + 1):
            if sk[j:j + len(p)] == p:
                return j, j + len(p) - 1
        return None
    a = find(start_phrase)
    if not a:
        return None
    first = a[0]
    if end_phrase:
        b = find(end_phrase, a[0])
        if not b:
            return None
        last = b[1]
    else:
        last = len(sk) - 1
    i0, i1 = seq[first][0], seq[last][0]
    text = " ".join(words[i0:i1 + 1])
    return ("… " if i0 > 0 else "") + text + (" …" if i1 < len(words) - 1 else "")

def select_tafsir(t, skip=0, n=None):
    sents = dedupe_sentences(t).split(" ")
    parts = re.split(r"(?<=[.؟!])\s+", dedupe_sentences(t))
    parts = parts[skip:]
    if n:
        parts = parts[:n]
    return " ".join(parts)

CALM_IDS = {"anger-7-200"}
HARM_WORDS = {"اذاني", "اذتني", "ظلمني", "ظلمتني", "اهانني", "اهانتني", "خانني", "خانتني", "جرحني", "جرحتني", "غدر", "غدروا", "سبني", "شتمني", "تنمر", "كذب", "كذبوا", "اساء", "اساؤوا", "يظلمني", "يهينني", "يسبني"}

class CuratedVerseEngine:
    def __init__(self, curated, banks):
        self.by_emo = curated["by_emotion"]
        self.banks = banks
        meta = {f"{emo}-{x['s']}-{x['a']}": x for emo, lst in CURATED_REFS.items() for x in lst}   # الوسوم من القائمة الحالية (مصدر واحد للحقيقة)
        self.fallbacks = []
        for emo in list(self.by_emo):
            kept = []
            for e in self.by_emo[emo]:
                m = meta.get(e["id"])
                if not m:
                    continue                      # آية حُذفت من القائمة الحالية
                e.update(forgive=m.get("forgive", False), trig=m["trig"], story=m["story"], narrow=m.get("narrow", False), notaf=m.get("notaf", False),
                         theme=m["theme"], gen=m["gen"], taf_skip=m.get("taf_skip", 0), taf_n=m.get("taf_n"))
                e["verse"] = strip_basmala(e["verse"])
                e["shown"] = e["verse"]
                if m.get("ex"):
                    cut = cut_excerpt(e["verse"], m["ex"][0], m["ex"][1])
                    if cut:
                        e["shown"] = cut
                    else:
                        self.fallbacks.append(e["id"])   # لم نجد المقطع: نعرض الآية كاملة
                kept.append(e)
            self.by_emo[emo] = kept
        for lst in self.by_emo.values():
            for e in lst:
                pass
        for lst in self.by_emo.values():
            for e in lst:
                e["trig_v"] = [v for t in e["trig"] for v in _variants(t)]
        self.recent = collections.defaultdict(lambda: collections.deque(maxlen=8))

    def stats(self):
        return {k: len(v) for k, v in self.by_emo.items()}

    def pick(self, emotion, user_text, user_key="default", calm_only=False):
        everything = [e for lst in self.by_emo.values() for e in lst]
        if emotion in self.by_emo:
            pool = self.by_emo[emotion]
        else:
            pool = everything                      # للحالة المحايدة: كل الآيات، لكن العامة منها فقط إلا إذا طابقت كلمة
        if calm_only:                              # عند وصف أذى: آية الاستعاذة/التهدئة فقط (لا عفو ولا أمر بالصبر على الأذى)
            calm = [e for e in pool if e["id"] in CALM_IDS]
            pool = calm or pool
        norm_text = clean_arabic_text(user_text)
        padded = " " + norm_text + " "
        generic = [e for e in pool if not e.get("narrow") and (emotion in self.by_emo or e.get("gen"))]
        hits = [e for e in pool if any(f" {v} " in padded for v in e["trig_v"])]
        if hits and len(hits) >= 3 and generic and random.random() < 0.3:
            cand = generic                         # عند وجود مواضيع كثيرة مطابقة نخلط بعض الآيات العامة للتنوع
        else:
            cand = hits or generic or pool
        fresh = [e for e in cand if e["id"] not in self.recent[user_key]]
        e = random.choice(fresh or cand)
        self.recent[user_key].append(e["id"])
        return e

    def compose(self, emotion, user_text, user_key="default", fem=False):
        harm = bool(set(clean_arabic_text(user_text).split()) & HARM_WORDS) and emotion in ("anger", "sadness", "fear", "neutral")
        e = self.pick(emotion, user_text, user_key, calm_only=(harm and emotion == "anger"))
        bank = self.banks["harm"] if harm else self.banks.get(emotion, self.banks["neutral"])
        parts = [random.choice(bank["open"])]
        if e["story"] and random.random() < 0.7:
            parts.append(e["story"])
        else:
            parts.append(random.choice(bank["bridge"]))
        parts.append(f"✨ ﴿ {simplify_verse(e.get('shown', e['verse']))} ﴾\n— {e['ref']}")
        taf = "" if e.get("notaf") else trim_tafsir(select_tafsir(e["tafsir"], e.get("taf_skip", 0), e.get("taf_n")))
        if taf:
            parts.append(f"📖 من التفسير الميسر: {taf}")
        parts.append(random.choice(bank["close"]))
        if random.random() < 0.5:
            parts.append("🤲 " + random.choice(bank["dua"]))
        return fill_gender("\n\n".join(parts), fem), e


In [ ]:
# الخلية 8: المعالجة الشاملة (طوارئ ← تحايا ← عاطفة [نموذج + كلمات صريحة] ← آية منتقاة ورد متنوع)
CONFIDENCE_THRESHOLD = 0.55
engine = CuratedVerseEngine(curated, BANKS)
print("المحرك جاهز:", engine.stats())
if engine.fallbacks:
    print("تنبيه: لم يُعثر على المقطع المقصود في هذه الآيات فتُعرض كاملة:", engine.fallbacks)

# تصدير ما يراه المستخدم فعلاً (الآية المعروضة + التفسير المعروض) لمراجعة المشرف
import pandas as pd
_rows = []
for _emo, _lst in engine.by_emo.items():
    for _e in _lst:
        _rows.append({"الشعور": _emo, "المرجع": _e["ref"], "الموضوع": _e["theme"], "الآية المعروضة": simplify_verse(_e["shown"]),
                      "التفسير المعروض": "" if _e["notaf"] else trim_tafsir(select_tafsir(_e["tafsir"], _e["taf_skip"], _e["taf_n"])),
                      "القصة": _e["story"], "كلمات التفعيل": " ، ".join(_e["trig"]), "سياق ضيق": _e["narrow"]})
pd.DataFrame(_rows).to_csv("/content/sanad_review_final.csv", index=False, encoding="utf-8-sig")
print("تم تصدير /content/sanad_review_final.csv للمراجعة")

# كلمات صريحة تعبّر عن الشعور (بعد التطبيع). النموذج وحده أخطأ كثيراً (يرجّح joy): فإذا ذكر المستخدم شعوره صراحةً نعتمد كلامه.
EMOTION_WORDS = {
    "anger": ["غضبان", "غضبانه", "غضب", "عصبي", "عصبيه", "اعصابي", "منفعل", "منفعله", "مستفز", "مستفزه"],
    "fear": ["خايف", "خايفه", "خوف", "قلق", "قلقان", "قلقانه", "توتر", "متوتر", "متوتره", "مرعوب", "مرعوبه", "رعب", "فزعان", "فزعانه"],
    "sadness": ["حزين", "حزينه", "حزن", "مهموم", "مهمومه", "مكتئب", "مكتئبه", "مخنوق", "مخنوقه"],
    "joy": ["فرحان", "فرحانه", "فرح", "سعيد", "سعيده", "مبسوط", "مبسوطه", "نجحت", "تخرجت", "مبروك"],
}
# مواقف صعبة واضحة: لا تُعامل كفرح مهما قال النموذج
HARDSHIP_WORDS = {
    "fear": ["محتاج", "محتاجه", "رزق", "فقر", "ديون", "ايجار", "عاطل", "عاطله", "بطاله", "مفلس", "مفلسه", "وظيفه"],
    "sadness": ["مات", "ماتت", "توفي", "توفيت", "وفاه", "فقدت", "مريض", "مريضه", "مرض", "وجع"],
}
_NEG = {"ما", "مش", "لا", "مو", "ماني"}
# حيرة/ضياع بدون شعور محدد: لا نفرض عليها خوفاً أو حزناً
CONFUSION_PHRASES = ["ما عارف", "ما عارفه", "مش عارف", "مش عارفه", "حيران", "حيرانه", "محتار", "محتاره", "ضايع", "ضايعه", "تايه", "تايهه", "مشتت", "مشتته"]

def _words_hit(tokens, words):
    hit = False
    for i, w in enumerate(tokens):
        if w in words and not (i > 0 and tokens[i - 1] in _NEG):
            hit = True
    return hit

# شعور منفي ("ما غضبانة"، "ما فرحانة"): النموذج بيغلط فيها بثقة عالية (اختبار الجمل الصعبة)، فنتعامل معاها بحذر
_NEG_EMO = set(w for ws in EMOTION_WORDS.values() for w in ws) | {"زعلان", "زعلانه", "مرتاح", "مرتاحه", "متضايق", "متضايقه"}
def _negated_emotion(tokens):
    return any(w in _NEG_EMO and i > 0 and tokens[i - 1] in _NEG for i, w in enumerate(tokens))

def decide_emotion(user_input):
    tokens = re.sub(r"[^\w\s]", " ", clean_arabic_text(user_input)).split()   # نشيل علامات الترقيم عشان "غضبانة،" تتعرف
    explicit = [e for e, ws in EMOTION_WORDS.items() if _words_hit(tokens, set(ws))]
    model_emotion, conf, _ = predict_emotion(user_input)
    if _negated_emotion(tokens) and not explicit:
        return "neutral", "negation-guard", model_emotion, conf
    emotion, source = model_emotion, "model"
    if conf < CONFIDENCE_THRESHOLD:
        emotion, source = "neutral", "low-confidence"
    if len(explicit) == 1 and explicit[0] != emotion:
        emotion, source = explicit[0], "explicit-word"
    padded = " " + " ".join(tokens) + " "
    if not explicit and any(f" {p} " in padded for p in CONFUSION_PHRASES):
        return "neutral", "confusion-phrase", model_emotion, conf
    if emotion in ("joy", "neutral") and "joy" not in explicit:
        for emo, ws in HARDSHIP_WORDS.items():
            if _words_hit(tokens, set(ws)):
                emotion, source = emo, "hardship-word"
                break
    return emotion, source, model_emotion, conf

def sanad_reply(user_input, user_key="default"):
    fem = detect_female(user_input)
    if fem:
        FEM_USERS.add(user_key)
    fem = fem or (user_key in FEM_USERS)
    # 1) الطوارئ أولاً (قبل التحايا)
    is_emg, category, msg = safety_layer.check(user_input)
    if is_emg:
        return {"path": "Safety Layer", "emotion": category, "response": fill_gender(msg, fem)}
    # 2) التحايا
    is_greet, greet = check_greetings(user_input)
    if is_greet:
        return {"path": "Greeting", "emotion": "neutral", "response": fill_gender(greet, fem)}
    # 3) العاطفة (نموذج + كلمات صريحة + مواقف صعبة)
    emotion, source, model_emotion, conf = decide_emotion(user_input)
    # 4) آية منتقاة + رد متنوع
    text, verse = engine.compose(emotion, user_input, user_key, fem)
    return {"path": "SANAD Curated", "emotion": emotion, "emotion_source": source, "model_said": model_emotion,
            "confidence": conf, "verse_id": verse["id"], "response": text}

المحرك جاهز: {'fear': 29, 'sadness': 25, 'anger': 22, 'joy': 23}
تم تصدير /content/sanad_review_final.csv للمراجعة


In [ ]:
# الخلية 9: اختبار سريع قبل البوت
import traceback
tests = ["السلام عليكم", "في حرامي في البيت وانا خايفة", "يا زول في حريق", "عايز اموت", "انا خايف من بكرة",
         "غضب شديد يعتريني", "انا حزينة جداً", "نجحت الحمد لله", "حصلت لي محادثة غريبة",
         "خايفة على ولدي", "مات ابوي الله يرحمه", "زوجي اذاني وانا غضبانة", "محتاج رزق"]
for t in tests:
    print("\n---", t)
    try:
        r = sanad_reply(t, user_key="test")
        print(f"[{r['path']} | {r['emotion']} ({r.get('emotion_source','-')}, النموذج قال: {r.get('model_said','-')}) | {r.get('verse_id','-')}]")
        print(r["response"][:600])
    except Exception:
        traceback.print_exc()


--- السلام عليكم
[Greeting | neutral (-, النموذج قال: -) | -]
أهلاً بيك يا زول يا طيب! حبابك حبابك، حاسس بشنو الليلة؟ فضفض لي أنا سامعك.. 🤍

--- في حرامي في البيت وانا خايفة
[Safety Layer | violence (-, النموذج قال: -) | -]
🚔 سلامتك الآن هي الأهم.
• لو في خطر مباشر، اتصلي بالشرطة (999) فوراً.
• اقفلي الأبواب، وابعدي عن مكان الخطر، وما تواجهي أي زول بنفسك.
• نادي الجيران أو أقرب زول تثق فيه.
ربنا يحفظك ويحفظ أهلك 🤍

--- يا زول في حريق
[Safety Layer | fire (-, النموذج قال: -) | -]
🔥 الأولوية: اطلعي من المكان فوراً.
• اتصلي بالإطفاء/الدفاع المدني (999).
• ما ترجعي تجيبي حاجات، وانتبهي من الدخان (ابقَي منخفضاً).
• نبّهي الناس حواليك.
ربنا يسلمك 🤍

--- عايز اموت
[Safety Layer | self_harm (-, النموذج قال: -) | -]
🤍 كلامك مهم عندي ومهتم بسلامتك.
لو في خطر إنك تؤذي نفسك الآن:
• اتصلي بالطوارئ (999) أو اذهبي لأقرب مستشفى فوراً.
• ما تبقي براك، نادي أقرب زول ليك (أهل، جيران، صديق) واقعدي معاه.
• ابعدي عن أي شيء ممكن تؤذي بيه نفسك.
أنا بوت وما أقدر أحل مكان مختص، لكن أنا هنا أسمعك، وتقدري تحكي ل

In [ ]:
# الخلية 10: بوت تيليجرام (التوكن من Colab Secrets باسم TELEGRAM_TOKEN، لا تكتبيه هنا أبداً)
import asyncio, traceback, nest_asyncio
from google.colab import userdata
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, ContextTypes, MessageHandler, filters

TOKEN = userdata.get("TELEGRAM_TOKEN")

async def start_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("أهلاً بك في بوت السند (المساعد الروحي والنفسي) 🤍\n"
                                    "اكتب لي ما يضايقك أو مشاعرك، وسأقوم بالرد عليك.")

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    try:
        reply = sanad_reply(update.message.text, user_key=str(update.effective_user.id))["response"]
    except Exception:
        traceback.print_exc()
        reply = "حدث خطأ أثناء معالجة النص، يرجى المحاولة لاحقاً."
    await update.message.reply_text(reply)

async def run_bot():
    app = ApplicationBuilder().token(TOKEN).build()
    app.add_handler(CommandHandler("start", start_command))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    print("⚡ البوت يعمل الآن...")
    await app.run_polling(poll_interval=1.0)

nest_asyncio.apply()
asyncio.run(run_bot())

⚡ البوت يعمل الآن...
